# DataPassport quickstart

Inspect a dataset, compare two versions, and check a contract. Run `python examples/make_data.py` first.

In [1]:
import polars as pl

import datapassport as dp

DATA = '../data'

## 1. Inspect

A passport is a fingerprint plus a column profile. It renders as a report in notebooks.

In [2]:
v1 = dp.inspect(f'{DATA}/orders_v1.csv', key='order_id')
v1

Passport(name='orders_v1', rows=5000, columns=7, fingerprint='7f0030f298ff')

The fingerprint ignores row order and file format. The Parquet copy and a shuffled frame match the CSV:

In [3]:
df = pl.read_parquet(f'{DATA}/orders_v1.parquet')
(v1.fingerprint.dataset == dp.fingerprint(df.sample(fraction=1, shuffle=True, seed=1)).dataset)

True

Column statistics as a Polars frame:

In [4]:
v1.column_table()

column,type,count,nulls,null_fraction,distinct,min,max,mean
str,str,i64,i64,f64,i64,f64,f64,f64
"""order_id""","""integer""",5000,0,0.0,5000,1.0,5000.0,2500.5
"""customer_email""","""string""",5000,0,0.0,1590,null,null,null
"""status""","""string""",5000,0,0.0,4,null,null,null
"""amount""","""float""",5000,0,0.0,3206,2.45,259.86,28.897264
"""quantity""","""integer""",5000,0,0.0,5,1.0,5.0,2.954
"""created_at""","""timestamp""",5000,0,0.0,4742,null,null,null
"""country""","""string""",5000,0,0.0,6,null,null,null


## 2. Compare versions

With a key, rows are matched and changes counted per column. Schema changes, including likely renames, and drift are reported.

In [5]:
diff = dp.compare(f'{DATA}/orders_v1.parquet', f'{DATA}/orders_v2.parquet', key='order_id')
print(diff.summary())
diff

+300 -100 ~100 rows; 1 schema change(s); 1 column(s) changed


Comparison(format='datapassport/comparison', format_version=1, base=VersionRef(name='orders_v1', uri='../data/orders_v1.parquet', fingerprint='7f0030f298ff718131b943e2881ee9d0b90d8eb307f76effc26770b147d481b8', rows=5000, created_at=datetime.datetime(2026, 11, 25, 6, 43, 32, tzinfo=datetime.timezone.utc)), target=VersionRef(name='orders_v2', uri='../data/orders_v2.parquet', fingerprint='b21f583e65c0e0d634f085acfec3253429faee213aa4798d3c47e65ee212039d', rows=5200, created_at=datetime.datetime(2026, 11, 25, 6, 43, 32, tzinfo=datetime.timezone.utc)), identical=False, schema_changes=[SchemaChange(kind='renamed', column='country', to_column='country_code', from_type='string', to_type='string', breaking=True, confidence='likely')], changed_columns=['amount'], rows=RowChanges(method='key', key=['order_id'], before=5000, after=5200, added=300, removed=100, changed=100, unchanged=4800, changed_by_column={'amount': 100}, examples=None, note=None), duplicates={'base': 0, 'target': 0}, drift={'customer_email': ColumnDrift(column='customer_email', kind='categorical', psi=0.09769801189775722, ks=None, wasserstein=None, wasserstein_norm=None, js=0.13385518634823135, tvd=0.021153846153846127, null_fraction_delta=0.0, mean_delta=None, status='pass', metric='psi', note=None), 'status': ColumnDrift(column='status', kind='categorical', psi=0.012828347609829122, ks=None, wasserstein=None, wasserstein_norm=None, js=0.04282747371219127, tvd=0.004369230769230795, null_fraction_delta=0.0, mean_delta=None, status='pass', metric='psi', note=None), 'amount': ColumnDrift(column='amount', kind='numeric', psi=0.008284373071586274, ks=0.034015748031496096, wasserstein=2.0307020000000016, wasserstein_norm=0.10041795030288055, js=None, tvd=None, null_fraction_delta=0.0, mean_delta=2.271284076923063, status='pass', metric='psi', note=None), 'quantity': ColumnDrift(column='quantity', kind='numeric', psi=0.0, ks=0.0, wasserstein=0.0, wasserstein_norm=0.0, js=0.0031000580231110414, tvd=0.003438461538461543, null_fraction_delta=0.0, mean_delta=0.0046538461538458975, status='pass', metric='psi', note=None), 'created_at': ColumnDrift(column='created_at', kind='temporal', psi=8.392930794218946e-05, ks=0.0037603792558312255, wasserstein=3050.946000003815, wasserstein_norm=0.002273474543120263, js=None, tvd=None, null_fraction_delta=0.0, mean_delta=None, status='pass', metric='psi', note=None)}, notes=[])

## 3. Check a contract

Draft one from v1, then check v2 against it with v1 as the baseline.

In [6]:
contract = dp.Contract.infer(v1)
print(contract.to_yaml()[:600])

dataset: orders_v1
key:
- order_id
rows:
  min: 1
columns:
  order_id:
    type: integer
    nullable: false
    unique: true
    min: 0
  customer_email:
    type: string
    nullable: false
  status:
    type: string
    nullable: false
  amount:
    type: float
    nullable: false
    min: 0
  quantity:
    type: integer
    nullable: false
    min: 0
  created_at:
    type: timestamp
    nullable: false
  country:
    type: string
    nullable: false
checks:
- rule: no_duplicate_rows
  severity: warn



In [7]:
report = dp.check(f'{DATA}/orders_v2.parquet', contract, baseline=v1)
print(report.status)
for r in report.failures + report.warnings:
    print(f'{r.status:5} {r.label}: {r.message}')

fail
fail  country: required: column 'country' is missing
fail  country: schema_renamed: renamed country → country_code (likely: similar values) (breaking)


## 4. Your own rule

In [8]:
@dp.rule('amount_under_1000')
def amount_under_1000(ctx):
    n = ctx.count_where('amount >= 1000')
    return dp.Outcome(passed=n == 0, observed=n, message=f'{n} large orders')

dp.check(f'{DATA}/orders_v2.parquet', {'checks': [{'rule': 'amount_under_1000'}]}).results

[CheckResult(rule='amount_under_1000', column=None, status='pass', severity='error', observed=0, expected=None, failing_rows=None, message='0 large orders')]